# Robust ASL CNN — Webcam-Ready Training
Trains a CNN with heavy augmentation to bridge the domain gap between
Sign MNIST (studio images) and real webcam input.
Exports TFLite → drop into `bonus/asl_model.tflite`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Install & Imports

In [ ]:
!pip install -q kaggle tensorflow

import os, math, random
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers

print('TF:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 2. Load Sign MNIST

In [ ]:
# Download if needed
if not os.path.exists('sign_mnist_train.csv'):
    os.environ['KAGGLE_USERNAME'] = 'abdullahashiry'
    os.environ['KAGGLE_KEY']      = 'KGAT_331632d901a6cb7a05431b55135bd8c2'
    !kaggle datasets download -d datamunge/sign-language-mnist --unzip

def load_csv(path):
    df     = pd.read_csv(path)
    labels = df['label'].values
    pixels = df.drop('label', axis=1).values.reshape(-1, 28, 28, 1).astype('float32') / 255.0
    return pixels, labels

train_path = ('sign_mnist_train/sign_mnist_train.csv'
              if os.path.exists('sign_mnist_train/sign_mnist_train.csv')
              else 'sign_mnist_train.csv')
test_path  = ('sign_mnist_test/sign_mnist_test.csv'
              if os.path.exists('sign_mnist_test/sign_mnist_test.csv')
              else 'sign_mnist_test.csv')

x_train, y_train = load_csv(train_path)
x_test,  y_test  = load_csv(test_path)

NUM_CLASSES = 25   # 26 letters minus J (motion) and Z (motion)
print(f'Train: {x_train.shape}  Test: {x_test.shape}')

## 3. Augmentation Pipeline
Each augmentation simulates a specific real-webcam degradation.

In [ ]:
# ── Elastic distortion (perspective warp simulation) ────────────────────────
def elastic_distort(img, alpha=8.0, sigma=3.0):
    """Random elastic warp — simulates perspective/lens distortion."""
    shape = img.shape[:2]
    dx = tf.random.normal(shape) * alpha
    dy = tf.random.normal(shape) * alpha

    # Smooth displacement fields
    dx = tf.expand_dims(tf.expand_dims(dx, 0), -1)
    dy = tf.expand_dims(tf.expand_dims(dy, 0), -1)
    kernel_size = int(6 * sigma + 1) | 1   # odd
    dx = tf.squeeze(tf.nn.avg_pool2d(dx, kernel_size, 1, 'SAME'))
    dy = tf.squeeze(tf.nn.avg_pool2d(dy, kernel_size, 1, 'SAME'))

    y_grid, x_grid = tf.meshgrid(
        tf.range(shape[0], dtype=tf.float32),
        tf.range(shape[1], dtype=tf.float32),
        indexing='ij'
    )
    ny = tf.clip_by_value(y_grid + dy, 0, shape[0] - 1)
    nx = tf.clip_by_value(x_grid + dx, 0, shape[1] - 1)

    indices = tf.stack([tf.cast(ny, tf.int32), tf.cast(nx, tf.int32)], axis=-1)
    return tf.gather_nd(img[..., 0], indices)[..., tf.newaxis]


# ── Random erasing (cutout) ─────────────────────────────────────────────────
def random_erasing(img, prob=0.5, min_frac=0.02, max_frac=0.25):
    """Randomly zero out a rectangular patch — simulates partial occlusion."""
    if tf.random.uniform(()) > prob:
        return img
    h, w = img.shape[:2]
    area = tf.cast(h * w, tf.float32)
    erase_area = tf.random.uniform((), min_frac, max_frac) * area
    ar    = tf.random.uniform((), 0.3, 3.3)
    eh    = tf.cast(tf.math.sqrt(erase_area / ar), tf.int32)
    ew    = tf.cast(tf.math.sqrt(erase_area * ar), tf.int32)
    eh    = tf.minimum(eh, h - 1)
    ew    = tf.minimum(ew, w - 1)
    top   = tf.random.uniform((), 0, h - eh, dtype=tf.int32)
    left  = tf.random.uniform((), 0, w - ew, dtype=tf.int32)
    mask  = tf.ones_like(img)
    patch = tf.zeros([eh, ew, 1])
    # Build mask with zeros in erased region
    paddings = [[top, h - top - eh], [left, w - left - ew], [0, 0]]
    patch_full = tf.pad(patch, paddings, constant_values=1.0)
    return img * patch_full


# ── Gaussian noise ──────────────────────────────────────────────────────────
def add_noise(img, stddev=0.05):
    noise = tf.random.normal(tf.shape(img), stddev=stddev)
    return tf.clip_by_value(img + noise, 0.0, 1.0)


# ── Full augmentation chain ─────────────────────────────────────────────────
augment_layer = keras.Sequential([
    layers.RandomRotation(0.15),           # ±27° — hand tilt
    layers.RandomZoom(0.20),               # ±20% scale — distance variation
    layers.RandomTranslation(0.10, 0.10),  # ±10% shift — off-center crop
    layers.RandomContrast(0.50),           # ±50% contrast — lighting variation
    layers.RandomBrightness(0.40),         # ±40% brightness — room lighting
], name='augmentation')

def augment(img, label):
    img = augment_layer(img, training=True)
    # Gaussian blur (simulate motion / low-quality webcam)
    if tf.random.uniform(()) < 0.4:
        img4 = tf.expand_dims(img, 0)                     # (1,28,28,1)
        img4 = tf.nn.avg_pool2d(img4, 3, 1, 'SAME')       # cheap blur proxy
        img  = tf.squeeze(img4, 0)
    # Gaussian noise
    if tf.random.uniform(()) < 0.5:
        img = add_noise(img, stddev=tf.random.uniform((), 0.01, 0.06))
    # Cutout
    img = random_erasing(img, prob=0.4)
    img = tf.clip_by_value(img, 0.0, 1.0)
    return img, label

print('Augmentation pipeline defined.')

## 4. Visualise Augmentations
Check augmented images look realistic before training.

In [ ]:
ALPHABET = {0:'A',1:'B',2:'C',3:'D',4:'E',5:'F',6:'G',7:'H',8:'I',
            10:'K',11:'L',12:'M',13:'N',14:'O',15:'P',16:'Q',17:'R',
            18:'S',19:'T',20:'U',21:'V',22:'W',23:'X',24:'Y'}

fig, axes = plt.subplots(4, 8, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    idx   = random.randint(0, len(x_train) - 1)
    img   = tf.constant(x_train[idx])
    aug, _ = augment(img, y_train[idx])
    ax.imshow(aug.numpy().squeeze(), cmap='gray', vmin=0, vmax=1)
    ax.set_title(ALPHABET.get(y_train[idx], '?'), fontsize=8)
    ax.axis('off')
plt.suptitle('Augmented training samples', y=1.01)
plt.tight_layout()
plt.show()

## 5. tf.data Pipeline

In [ ]:
BATCH_SIZE = 128
AUTOTUNE   = tf.data.AUTOTUNE

ds_train = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .shuffle(len(x_train), seed=42)
    .map(augment, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

ds_test = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print(f'Train batches: {len(ds_train)}  Test batches: {len(ds_test)}')

## 6. Model Architecture
Deeper CNN with residual connections and global average pooling.
Keeps input 28×28×1 — same as current TFLite model, no Livestream.py changes needed.

In [ ]:
def residual_block(x, filters, stride=1):
    """Conv → BN → ReLU → Conv → BN + skip → ReLU"""
    shortcut = x
    x = layers.Conv2D(filters, 3, stride, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(filters, 3, 1,      padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    # Match dimensions if needed
    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, stride, use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
    x = layers.Add()([x, shortcut])
    x = layers.ReLU()(x)
    return x


def build_robust_cnn(num_classes=25):
    inp = keras.Input(shape=(28, 28, 1), name='input')

    # Stem
    x = layers.Conv2D(32, 3, padding='same', use_bias=False)(inp)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    # Stage 1 — 28×28
    x = residual_block(x, 32)
    x = residual_block(x, 32)

    # Stage 2 — 14×14
    x = residual_block(x, 64, stride=2)
    x = residual_block(x, 64)
    x = layers.Dropout(0.2)(x)

    # Stage 3 — 7×7
    x = residual_block(x, 128, stride=2)
    x = residual_block(x, 128)
    x = layers.Dropout(0.2)(x)

    # Stage 4 — 4×4
    x = residual_block(x, 256, stride=2)
    x = layers.Dropout(0.3)(x)

    # Head — Global Average Pooling (spatial invariance, fewer params than Flatten)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    out = layers.Dense(num_classes, activation='softmax', name='predictions')(x)

    return keras.Model(inp, out, name='RobustASL')


model = build_robust_cnn()
model.summary()
print(f'\nParam count: {model.count_params():,}')

## 7. Train

In [ ]:
EPOCHS    = 60
LR_INIT   = 3e-3
LR_MIN    = 1e-5

# Cosine decay with warm restarts
lr_schedule = keras.optimizers.schedules.CosineDecayRestarts(
    initial_learning_rate=LR_INIT,
    first_decay_steps=len(ds_train) * 10,  # restart every 10 epochs
    t_mul=1.5,
    m_mul=0.9,
    alpha=LR_MIN,
)

model.compile(
    optimizer=keras.optimizers.Adam(lr_schedule),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        'best_robust_cnn.keras',
        monitor='val_accuracy', save_best_only=True, verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=12, restore_best_weights=True, verbose=1
    ),
]

history = model.fit(
    ds_train,
    epochs=EPOCHS,
    validation_data=ds_test,
    callbacks=callbacks,
)

## 8. Evaluate

In [ ]:
loss, acc = model.evaluate(ds_test, verbose=0)
print(f'Test accuracy: {acc*100:.2f}%')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['accuracy'],     label='train')
ax1.plot(history.history['val_accuracy'], label='val')
ax1.set_title('Accuracy'); ax1.legend()
ax2.plot(history.history['loss'],     label='train')
ax2.plot(history.history['val_loss'], label='val')
ax2.set_title('Loss'); ax2.legend()
plt.tight_layout()
plt.show()

## 9. Export TFLite
Exports Float32 TFLite — drop `asl_model_robust.tflite` into `bonus/` and rename to `asl_model.tflite`.

In [ ]:
# Float32 TFLite
converter    = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
with open('asl_model_robust.tflite', 'wb') as f:
    f.write(tflite_model)

# INT8 quantised (smaller, faster on Pi)
def rep_dataset():
    for i in range(min(300, len(x_test))):
        yield [x_test[i:i+1]]

conv_int8 = tf.lite.TFLiteConverter.from_keras_model(model)
conv_int8.optimizations = [tf.lite.Optimize.DEFAULT]
conv_int8.representative_dataset = rep_dataset
conv_int8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
conv_int8.inference_input_type  = tf.float32
conv_int8.inference_output_type = tf.float32
tflite_int8 = conv_int8.convert()
with open('asl_model_robust_int8.tflite', 'wb') as f:
    f.write(tflite_int8)

print(f'FP32: {os.path.getsize("asl_model_robust.tflite")/1e6:.2f} MB')
print(f'INT8: {os.path.getsize("asl_model_robust_int8.tflite")/1e6:.2f} MB')

## 10. Copy to Drive

In [ ]:
import shutil

DRIVE_OUT = '/content/drive/MyDrive/CV552_SignLanguage/tflite'
os.makedirs(DRIVE_OUT, exist_ok=True)

shutil.copy('best_robust_cnn.keras',        os.path.join(DRIVE_OUT, 'robust_cnn.keras'))
shutil.copy('asl_model_robust.tflite',      os.path.join(DRIVE_OUT, 'asl_model_robust.tflite'))
shutil.copy('asl_model_robust_int8.tflite', os.path.join(DRIVE_OUT, 'asl_model_robust_int8.tflite'))
print('Saved to Drive.')